# Round 4 — Spread Distribution & Mean ± 1 Signal Hunt

**Products in scope:**
- `HYDROGEL_PACK`
- `VELVETFRUIT_EXTRACT`
- `VEV_4500`, `VEV_5000`, `VEV_5100`, `VEV_5200`, `VEV_5300`

**What we're looking for** (from prior-round signal): the L1 bid–ask spread for many Prosperity instruments
is a **discrete distribution centered on a few integer values**. The mode is usually the market-maker's
comfort zone. Specific spread values (e.g. `mode + 1`) often correlate with regime changes — informed
flow widens the book, while passive periods compress it.

We'll measure:
1. The full spread distribution per product.
2. Frequency at `mode`, `mode ± 1`, `mode ± 2`.
3. Time-series of spread vs mid price, with `mean ± 1σ` bands.
4. **Forward-return conditioning**: given spread = X right now, what's the next-tick mid move?
5. Counterparty signature in each spread regime — which Mark shows up when?
6. Cross-product spread correlation.

## 0. Setup

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

DATA_DIR = 'Dataset/ROUND_4'
PRODUCTS = ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT',
            'VEV_4500', 'VEV_5000', 'VEV_5100', 'VEV_5200', 'VEV_5300']

In [ ]:
def load():
    pframes = [pd.read_csv(f, delimiter=';') for f in sorted(glob.glob(f'{DATA_DIR}/prices_*.csv'))]
    tframes = []
    for f in sorted(glob.glob(f'{DATA_DIR}/trades_*.csv')):
        df = pd.read_csv(f, delimiter=';')
        m = re.search(r'day_(\d+)', f); df['day'] = int(m.group(1)) if m else 0
        tframes.append(df)
    p = pd.concat(pframes, ignore_index=True)
    t = pd.concat(tframes, ignore_index=True)
    p['spread'] = p['ask_price_1'] - p['bid_price_1']
    p['time_s'] = p['timestamp'] / 100
    p['t_global'] = p['day'] * 1_000_000 + p['timestamp']
    t['buyer'] = t['buyer'].fillna('').astype(str).str.strip()
    t['seller'] = t['seller'].fillna('').astype(str).str.strip()
    t['t_global'] = t['day'] * 1_000_000 + t['timestamp']
    return p, t

prices, trades = load()
print(f'prices: {len(prices):,} rows, {prices["product"].nunique()} products, {prices["day"].nunique()} days')
print(f'trades: {len(trades):,} rows, {len(set(trades["buyer"]).union(trades["seller"]) - {"", "UNKNOWN"})} counterparties')

## 1. Spread distribution per product

How discrete is each product's spread?  What's the mode?

In [ ]:
summary_rows = []
for prod in PRODUCTS:
    s = prices.loc[prices['product'] == prod, 'spread'].dropna().astype(int)
    if s.empty:
        continue
    summary_rows.append({
        'product': prod,
        'n': len(s),
        'min': int(s.min()), 'max': int(s.max()),
        'mean': float(s.mean()), 'std': float(s.std()),
        'median': float(s.median()),
        'mode': int(s.mode().iloc[0]),
        'n_unique_values': int(s.nunique()),
    })
spread_summary = pd.DataFrame(summary_rows)
spread_summary

In [ ]:
# Frequency of every observed spread value, per product (compact pivot)
freq = (prices[prices['product'].isin(PRODUCTS)]
        .assign(spread=lambda d: d['spread'].astype('Int64'))
        .groupby(['product', 'spread']).size()
        .unstack(fill_value=0))
# normalize per row → fraction of time at each spread
freq_pct = freq.div(freq.sum(axis=1), axis=0) * 100
freq_pct.round(2)

In [ ]:
# Distribution histograms: subplot grid, one per product
fig = make_subplots(rows=2, cols=4, subplot_titles=PRODUCTS, vertical_spacing=0.18)
for i, prod in enumerate(PRODUCTS):
    s = prices.loc[prices['product'] == prod, 'spread'].dropna().astype(int)
    if s.empty:
        continue
    r, c = (i // 4) + 1, (i % 4) + 1
    counts = s.value_counts().sort_index()
    fig.add_trace(go.Bar(x=counts.index, y=counts.values, marker_color='#38bdf8',
                         hovertemplate=f'{prod}<br>spread=%{{x}}<br>count=%{{y}}<extra></extra>'),
                  row=r, col=c)
    mode_v = int(s.mode().iloc[0])
    fig.add_vline(x=mode_v, line_color='#fbbf24', line_dash='dash', row=r, col=c)
fig.update_layout(template='plotly_dark', height=600, showlegend=False,
                  title='Spread distribution by product (mode marked)')
fig.show()

## 2. The mean ± 1 frequency table

For each product, how often does the spread sit at
`mode − 2`, `mode − 1`, `mode`, `mode + 1`, `mode + 2`?

In [ ]:
rows = []
for prod in PRODUCTS:
    s = prices.loc[prices['product'] == prod, 'spread'].dropna().astype(int)
    if s.empty:
        continue
    n = len(s)
    mode_v = int(s.mode().iloc[0])
    mean_v = float(s.mean())
    std_v = float(s.std())
    rows.append({
        'product': prod,
        'mean': round(mean_v, 2),
        'std': round(std_v, 2),
        'mode': mode_v,
        'pct_mode-2': round((s == mode_v - 2).mean() * 100, 2),
        'pct_mode-1': round((s == mode_v - 1).mean() * 100, 2),
        'pct_mode':   round((s == mode_v    ).mean() * 100, 2),
        'pct_mode+1': round((s == mode_v + 1).mean() * 100, 2),
        'pct_mode+2': round((s == mode_v + 2).mean() * 100, 2),
        'pct_other':  round(((s < mode_v - 2) | (s > mode_v + 2)).mean() * 100, 2),
    })
freq_table = pd.DataFrame(rows)
freq_table

In [ ]:
# Visualize the mean ± 1 frequency as a stacked horizontal bar
view = freq_table.set_index('product')[['pct_mode-2', 'pct_mode-1', 'pct_mode', 'pct_mode+1', 'pct_mode+2', 'pct_other']]
fig = go.Figure()
palette = ['#7c3aed', '#3b82f6', '#fbbf24', '#fb923c', '#f87171', '#64748b']
for col, color in zip(view.columns, palette):
    fig.add_trace(go.Bar(name=col, y=view.index, x=view[col], orientation='h',
                         marker_color=color,
                         hovertemplate=f'{col}<br>%{{y}}: %{{x:.2f}}%<extra></extra>'))
fig.update_layout(barmode='stack', template='plotly_dark', height=420,
                  title='% of ticks at each spread value relative to the mode',
                  xaxis_title='% of ticks', yaxis_title='Product')
fig.show()

## 3. Spread time-series with `mean ± 1` bands

Wide bands ⇒ the book regularly spikes wider than usual; that's where signal often hides.

In [ ]:
fig = make_subplots(rows=len(PRODUCTS), cols=1, shared_xaxes=False,
                    subplot_titles=PRODUCTS, vertical_spacing=0.04)
for i, prod in enumerate(PRODUCTS, start=1):
    p = prices[prices['product'] == prod].sort_values('t_global')
    if p.empty:
        continue
    s = p['spread'].astype(float)
    m, sd = float(s.mean()), float(s.std())
    fig.add_trace(go.Scatter(x=p['t_global'], y=s, mode='lines',
                             line=dict(color='#38bdf8', width=0.8),
                             name=prod, showlegend=False), row=i, col=1)
    fig.add_hline(y=m,      line_color='#fbbf24', line_dash='dash', line_width=1, row=i, col=1)
    fig.add_hline(y=m + 1,  line_color='#f87171', line_dash='dot',  line_width=1, row=i, col=1)
    fig.add_hline(y=m - 1,  line_color='#34d399', line_dash='dot',  line_width=1, row=i, col=1)
fig.update_layout(template='plotly_dark', height=180 * len(PRODUCTS), showlegend=False,
                  title='Spread time-series with mean (yellow) and mean ± 1 (green / red)')
fig.show()

## 4. Forward-return conditioning  —  the signal hunt

**Hypothesis:** specific spread values precede directional moves. For each product, group ticks by their
spread and compute the *mean forward return* of the mid at horizons `+1, +5, +20, +50` ticks.

If random, the mean forward return is ≈ 0. If a spread value reliably predicts direction, you'll see
a clean non-zero mean (with enough N to be significant).

In [ ]:
HORIZONS = [1, 5, 20, 50]

def fwd_return_table(product, k_values=(-2, -1, 0, 1, 2)):
    p = prices[prices['product'] == product].sort_values(['day', 'timestamp']).reset_index(drop=True)
    if p.empty:
        return pd.DataFrame()
    p = p.copy()
    mode_v = int(p['spread'].dropna().mode().iloc[0])
    # forward mids per day
    rows = []
    for d, dfp in p.groupby('day'):
        dfp = dfp.set_index('timestamp')['mid_price']
        for ts, mid in dfp.items():
            sp = int(p.loc[(p['day'] == d) & (p['timestamp'] == ts), 'spread'].iloc[0])
            rel = sp - mode_v
            if rel not in k_values:
                continue
            r = {'rel': rel}
            for k in HORIZONS:
                fwd = dfp.get(ts + 100 * k, np.nan)
                r[f'fwd_{k}'] = (fwd - mid) if pd.notna(fwd) else np.nan
            rows.append(r)
    if not rows:
        return pd.DataFrame()
    rdf = pd.DataFrame(rows)
    agg = rdf.groupby('rel').agg(['mean', 'std', 'count']).round(4)
    agg.index = [f'mode{r:+d}' if r != 0 else 'mode' for r in agg.index]
    return agg

for prod in PRODUCTS:
    print(f'\n=== {prod} (mode={int(prices.loc[prices["product"] == prod, "spread"].mode().iloc[0])}) ===')
    tbl = fwd_return_table(prod)
    if tbl.empty:
        print('  no data')
        continue
    print(tbl)

In [ ]:
# Visualize: heatmap of mean forward return by (product, spread relative to mode, horizon)
blocks = []
for prod in PRODUCTS:
    p = prices[prices['product'] == prod].sort_values(['day','timestamp']).reset_index(drop=True)
    if p.empty:
        continue
    mode_v = int(p['spread'].dropna().mode().iloc[0])
    p = p.copy()
    for k in HORIZONS:
        p[f'fwd_{k}'] = (p.groupby('day')['mid_price'].shift(-k) - p['mid_price'])
    p['rel'] = p['spread'].astype(int) - mode_v
    p = p[p['rel'].between(-2, 2)]
    if p.empty:
        continue
    means = p.groupby('rel')[[f'fwd_{k}' for k in HORIZONS]].mean()
    means['product'] = prod
    means = means.reset_index()
    blocks.append(means)
if blocks:
    big = pd.concat(blocks, ignore_index=True)
    big_long = big.melt(id_vars=['product', 'rel'],
                        value_vars=[f'fwd_{k}' for k in HORIZONS],
                        var_name='horizon', value_name='mean_fwd')
    big_long['horizon'] = big_long['horizon'].str.replace('fwd_', 'k=', regex=False)
    big_long['cell'] = big_long['product'] + '  rel=' + big_long['rel'].astype(str)
    pivot = big_long.pivot(index='cell', columns='horizon', values='mean_fwd')
    pivot = pivot.loc[[c for prod in PRODUCTS for c in pivot.index if c.startswith(prod)]]
    fig = go.Figure(data=go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index,
                                     colorscale='RdBu', zmid=0,
                                     hovertemplate='%{y}<br>%{x}<br>mean fwd=%{z:+.3f}<extra></extra>'))
    fig.update_layout(template='plotly_dark', height=900,
                      title='Mean forward mid-price move conditional on spread − mode  (red = up move follows)')
    fig.show()

## 5. Counterparty signature in each spread regime

Which Marks show up when spread is at `mode`, `mode + 1`, etc.? If a specific Mark always appears
when the spread is at a non-mode value, that Mark is part of why the spread widens.

In [ ]:
# Build (timestamp, product) -> spread lookup
key_to_spread = (prices[['day', 'timestamp', 'product', 'spread']]
                 .dropna()
                 .rename(columns={'product': 'symbol'}))
trades_aug = trades.merge(key_to_spread, on=['day', 'timestamp', 'symbol'], how='left')

for prod in PRODUCTS:
    tdf = trades_aug[trades_aug['symbol'] == prod].dropna(subset=['spread']).copy()
    if tdf.empty:
        continue
    mode_v = int(prices.loc[prices['product'] == prod, 'spread'].dropna().mode().iloc[0])
    tdf['rel'] = tdf['spread'].astype(int) - mode_v
    tdf['mark'] = np.where(tdf['buyer'].isin(['', 'UNKNOWN']),
                            tdf['seller'],
                            np.where(tdf['seller'].isin(['', 'UNKNOWN']),
                                     tdf['buyer'], tdf['buyer']))
    counts = (tdf.assign(side=np.where(tdf['rel'] == 0, 'mode',
                                        np.where(tdf['rel'] < 0, '<mode',
                                                 '>mode')))
                  .groupby(['mark', 'side']).size().unstack(fill_value=0))
    if counts.empty:
        continue
    print(f'\n=== {prod} (mode spread = {mode_v}) — trades by Mark, by spread regime ===')
    print(counts.sort_values(by=counts.columns.tolist(), ascending=False))

## 6. Cross-product spread correlation

When the book widens on one product, does it widen on another?

In [ ]:
wide = (prices[prices['product'].isin(PRODUCTS)]
        .pivot_table(index=['day', 'timestamp'], columns='product', values='spread', aggfunc='last')
        .ffill())
corr = wide.corr().round(3)
fig = go.Figure(data=go.Heatmap(z=corr.values, x=corr.columns, y=corr.index,
                                 colorscale='RdBu', zmid=0,
                                 hovertemplate='corr(%{y}, %{x}) = %{z:.3f}<extra></extra>'))
fig.update_layout(template='plotly_dark', height=520,
                  title='Cross-product spread correlation (Pearson)')
fig.show()
corr

## 7. Findings template

Use this section to record what shows up in the conditional forward-return table and the counterparty regime table.
Specifically check:
- For each product, is `mean fwd` at `rel = +1` (or `−1`) clearly non-zero? If yes, the spread itself is a signal.
- Does a specific Mark (e.g. Mark 22 / Mark 67) preferentially trade at `rel ≠ 0`? That's a regime they create.
- Are the voucher spreads correlated to underlying spread? If yes, vouchers can be modeled by underlying-spread features.

Notes:
